<a href="https://www.kaggle.com/code/avikdas567/predict-1-year-us-stock-returns-from-fundamentals?scriptVersionId=311248372" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import catboost as cb

SEED = 42
TARGET = "return_pct"

# =========================
# LOAD DATA
# =========================
train = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/train.csv")
test = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/predict-1-year-us-stock-returns-from-fundamentals/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =========================
# PREPROCESS
# =========================
def preprocess(df):
    df = df.copy()
    
    if "period_start" in df.columns:
        df["period_start"] = pd.to_datetime(df["period_start"])
        df["year"] = df["period_start"].dt.year
    else:
        df["year"] = 2024
    
    return df

train = preprocess(train)
test = preprocess(test)

# =========================
# FEATURE ENGINEERING
# =========================
def feature_engineering(df):
    df = df.copy()
    
    df["profit_to_assets"] = df["net_income_ttm"] / (df["total_assets"] + 1e-6)
    df["debt_ratio"] = df["long_term_debt"] / (df["total_assets"] + 1e-6)
    df["asset_turnover"] = df["revenue_ttm"] / (df["total_assets"] + 1e-6)
    
    df["valuation_efficiency"] = df["roe"] / (df["pe_ttm"] + 1e-6)
    df["growth_adjusted_pe"] = df["pe_ttm"] / (df["revenue_growth_yoy"] + 1e-6)
    
    # 🔥 NEW INTERACTIONS (HIGH IMPACT)
    df["roe_pe"] = df["roe"] / (df["pe_ttm"] + 1e-6)
    df["margin_growth"] = df["net_margin"] * df["revenue_growth_yoy"]
    df["debt_profit"] = df["debt_ratio"] * df["profit_to_assets"]
    
    return df

train = feature_engineering(train)
test = feature_engineering(test)

# =========================
# SECTOR + YEAR RANK FEATURES
# =========================
def add_rank_features(df):
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in num_cols:
        if col not in ["id", TARGET]:
            df[f"{col}_rank"] = df.groupby(["year", "sector_code"])[col].rank(pct=True)
    
    return df

train = add_rank_features(train)
test = add_rank_features(test)

# =========================
# TARGET TRANSFORM
# =========================
train[TARGET] = np.clip(train[TARGET], -100, 200)
y_original = train[TARGET].copy()

train[TARGET] = np.sign(train[TARGET]) * np.log1p(np.abs(train[TARGET]))

# =========================
# FEATURES
# =========================
drop_cols = ["id", "ticker", "period_start", "period_end", TARGET]
features = [c for c in train.columns if c not in drop_cols]

# =========================
# MISSING VALUES
# =========================
for col in features:
    median = train[col].median()
    train[col] = train[col].fillna(median)
    test[col] = test[col].fillna(median)

# =========================
# SCALING
# =========================
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train[features])
test_scaled = scaler.transform(test[features])

# =========================
# TIME-BASED CV
# =========================
years = sorted(train["year"].unique())

folds = []
for i in range(len(years) - 1):
    tr_idx = train[train["year"] <= years[i]].index
    val_idx = train[train["year"] == years[i+1]].index
    folds.append((tr_idx, val_idx))

# =========================
# STORAGE
# =========================
oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
oof_ridge = np.zeros(len(train))

pred_lgb = np.zeros(len(test))
pred_cat = np.zeros(len(test))
pred_ridge = np.zeros(len(test))

# =========================
# TRAIN LOOP
# =========================
for fold, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== TIME FOLD {fold+1} =====")
    
    X_tr, X_val = train.loc[tr_idx, features], train.loc[val_idx, features]
    y_tr, y_val = train.loc[tr_idx, TARGET], train.loc[val_idx, TARGET]
    
    # LightGBM (MAIN MODEL)
    lgb_model = lgb.LGBMRegressor(
        n_estimators=900,
        learning_rate=0.02,
        num_leaves=64,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=SEED,
        device="gpu"
    )
    
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    pred_lgb += lgb_model.predict(test[features]) / len(folds)
    
    # CatBoost (SUPPORT MODEL)
    cat_model = cb.CatBoostRegressor(
        iterations=600,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=0
    )
    
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=0)
    
    oof_cat[val_idx] = cat_model.predict(X_val)
    pred_cat += cat_model.predict(test[features]) / len(folds)
    
    # Ridge (STABILITY)
    ridge = Ridge(alpha=10)
    ridge.fit(train_scaled[tr_idx], y_tr)
    
    oof_ridge[val_idx] = ridge.predict(train_scaled[val_idx])
    pred_ridge += ridge.predict(test_scaled) / len(folds)

# =========================
# ENSEMBLE
# =========================
oof = 0.6 * oof_lgb + 0.3 * oof_cat + 0.1 * oof_ridge
pred = 0.6 * pred_lgb + 0.3 * pred_cat + 0.1 * pred_ridge

# =========================
# STABILITY FIX
# =========================
oof = np.clip(oof, -5, 5)
pred = np.clip(pred, -5, 5)

oof = np.sign(oof) * (np.expm1(np.abs(oof)))
pred = np.sign(pred) * (np.expm1(np.abs(pred)))

oof = np.nan_to_num(oof, nan=0.0, posinf=200, neginf=-100)
pred = np.nan_to_num(pred, nan=0.0, posinf=200, neginf=-100)

# =========================
# CV SCORE
# =========================
rmse = np.sqrt(mean_squared_error(y_original, oof))
print("\nCV RMSE:", rmse)

# =========================
# FINAL SUBMISSION
# =========================
pred = np.clip(pred, -50, 150)

submission = sample.copy()
submission["return_pct"] = pred
submission.to_csv("submission.csv", index=False)

print("\n'submission.csv' created!")

Train shape: (23070, 39)
Test shape: (8520, 36)

===== TIME FOLD 1 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 20039
[LightGBM] [Info] Number of data points in the train set: 5029, number of used features: 84
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 80 dense feature groups (0.38 MB) transferred to GPU in 0.001180 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -0.523252

===== TIME FOLD 2 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 20200
[LightGBM] [Info] Number of data points in the train set: 10368, number of used features: 86
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 82 dense feature groups (0.83 MB) transferred to GPU in 0.001247 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 1.326807

===== TIME FOLD 3 =====
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 20389
[LightGBM] [Info] Number of data points in